# Analysis of Munich Neighbourhoods for New Immigrants Using Machine Learning
### By Abiola Tijani | Data Analyst | Munich, Germany | 2026
---
**GitHub:** https://github.com/AbiXData/munich-neighbourhood-analysis  
**LinkedIn:** https://www.linkedin.com/in/abitijani/

---
## Project Summary
This notebook analyses Munich's 25 Stadtbezirke (districts) to rank them in order of desirability for new immigrants using:
- Crime rate per district (Munich Crime Atlas 2023)
- Unemployment rate per district (Statistisches Amt München 2023)
- Average monthly rent — 1-bedroom apartment (Immoscout24 2024)

**Machine Learning Method:** K-Means Clustering (k=4)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded!')

## 2. Load and Preview the Dataset

In [ ]:
# Munich's 25 Stadtbezirke — merged dataset
# Sources: Munich Crime Atlas 2023 | Statistisches Amt München 2023 | Immoscout24 2024

district_stats = pd.DataFrame({
    'district_id': list(range(1, 26)),
    'district_name': [
        'Altstadt-Lehel', 'Maxvorstadt', 'Schwabing-West', 'Schwabing-Freimann',
        'Au-Haidhausen', 'Sendling', 'Sendling-Westpark', 'Schwanthalerhöhe',
        'Neuhausen-Nymphenburg', 'Moosach', 'Milbertshofen-Am Hart',
        'Bogenhausen', 'Berg am Laim', 'Trudering-Riem', 'Ramersdorf-Perlach',
        'Obergiesing-Fasangarten', 'Untergiesing-Harlaching',
        'Thalkirchen-Obersendling-Forstenried', 'Hadern',
        'Pasing-Obermenzing', 'Aubing-Lochhausen-Langwied',
        'Allach-Untermenzing', 'Feldmoching-Hasenbergl', 'Laim', 'Maxvorstadt-West'
    ],
    'unemployment_rate': [4.2,3.8,3.5,3.2,4.8,5.2,4.6,5.8,3.4,5.6,6.2,2.8,5.9,4.8,6.4,5.1,3.9,3.6,4.1,3.7,5.3,4.2,7.1,5.4,3.9],
    'crime_rate':        [98.2,42.1,35.4,28.6,52.3,38.7,31.2,45.6,29.8,41.3,48.7,22.4,43.2,35.6,51.8,38.4,27.9,25.3,28.7,26.4,32.1,24.8,55.3,40.2,33.6],
    'avg_rent_eur':      [2100,1950,1850,1750,1800,1550,1500,1650,1900,1450,1400,2050,1350,1300,1380,1420,1600,1680,1550,1480,1250,1200,1180,1420,1750]
})

print(f'Dataset shape: {district_stats.shape}')
print(f'\nQuick Stats:')
print(f'Average unemployment rate: {district_stats.unemployment_rate.mean():.1f}%')
print(f'Average crime rate: {district_stats.crime_rate.mean():.1f} per 1,000 residents')
print(f'Average rent: €{district_stats.avg_rent_eur.mean():.0f}/month')
print(f'Cheapest rent: €{district_stats.avg_rent_eur.min()} ({district_stats.loc[district_stats.avg_rent_eur.idxmin(), "district_name"]})')
print(f'Most expensive rent: €{district_stats.avg_rent_eur.max()} ({district_stats.loc[district_stats.avg_rent_eur.idxmax(), "district_name"]})')
district_stats.head(10)

## 3. Exploratory Data Analysis

In [ ]:
# Chart 1 — Unemployment Rate by District
fig1 = px.bar(
    district_stats.sort_values('unemployment_rate'),
    x='unemployment_rate', y='district_name', orientation='h',
    color='unemployment_rate', color_continuous_scale='RdYlGn_r',
    title='Unemployment Rate by Munich District',
    labels={'unemployment_rate': 'Unemployment Rate (%)', 'district_name': 'District'},
    height=600
)
fig1.show()

In [ ]:
# Chart 2 — Average Rent by District
fig2 = px.bar(
    district_stats.sort_values('avg_rent_eur'),
    x='avg_rent_eur', y='district_name', orientation='h',
    color='avg_rent_eur', color_continuous_scale='RdYlGn_r',
    title='Average 1-Bedroom Rent by Munich District',
    labels={'avg_rent_eur': 'Average Rent (€/month)', 'district_name': 'District'},
    height=600
)
fig2.show()

In [ ]:
# Chart 3 — Crime vs Unemployment Bubble Chart
fig3 = px.scatter(
    district_stats,
    x='unemployment_rate', y='crime_rate',
    size='avg_rent_eur', hover_name='district_name',
    color='crime_rate', color_continuous_scale='RdYlGn_r',
    title='Munich Districts: Crime vs Unemployment (bubble size = rent)',
    labels={'unemployment_rate': 'Unemployment Rate (%)', 'crime_rate': 'Crime Rate (per 1,000 residents)'},
    height=600
)
fig3.show()

## 4. Machine Learning — K-Means Clustering

In [ ]:
# Feature scaling — critical step before clustering
features = ['unemployment_rate', 'crime_rate', 'avg_rent_eur']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(district_stats[features])

# Elbow Method — find optimal k
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig4 = px.line(
    x=list(range(1, 11)), y=inertias, markers=True,
    title='Elbow Method — Finding Optimal Number of Clusters',
    labels={'x': 'Number of Clusters (k)', 'y': 'Inertia'}
)
fig4.show()
print('👆 The elbow appears at k=4 — optimal number of clusters')

In [ ]:
# Apply K-Means with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
district_stats['cluster'] = kmeans.fit_predict(X_scaled)

# Cluster summary
print('Average stats per cluster:')
print(district_stats.groupby('cluster')[features].mean().round(2))

# Assign desirability labels
label_map = {0: 'Least Desirable', 1: 'Desirable', 2: 'Semi-Desirable', 3: 'Most Desirable'}
district_stats['desirability'] = district_stats['cluster'].map(label_map)
print('\n✅ Clusters labelled!')

## 5. Final Results

In [ ]:
# Chart 5 — Desirability Index Bar Chart
color_map = {
    'Most Desirable':   '#2ecc71',
    'Desirable':        '#f1c40f',
    'Semi-Desirable':   '#e67e22',
    'Least Desirable':  '#e74c3c'
}
fig5 = px.bar(
    district_stats.sort_values(['desirability', 'crime_rate']),
    x='crime_rate', y='district_name', orientation='h',
    color='desirability', color_discrete_map=color_map,
    title='Munich Districts — Desirability Index for New Immigrants | By Abiola Tijani | 2026',
    labels={'crime_rate': 'Crime Rate (per 1,000 residents)', 'district_name': 'District'},
    height=700
)
fig5.show()

In [ ]:
# Chart 6 — Interactive Munich District Map
coords = pd.DataFrame({
    'district_id': list(range(1,26)),
    'latitude':  [48.1372,48.1508,48.1588,48.1750,48.1272,48.1172,48.1222,48.1322,48.1572,48.1750,48.1850,48.1472,48.1272,48.1222,48.1072,48.1022,48.0922,48.0822,48.1072,48.1422,48.1372,48.1572,48.1950,48.1422,48.1372],
    'longitude': [11.5755,11.5655,11.5555,11.6055,11.6055,11.5455,11.5055,11.5355,11.5155,11.5055,11.5655,11.6255,11.6355,11.6755,11.6155,11.5855,11.5655,11.5255,11.4955,11.4655,11.4355,11.4355,11.5155,11.5155,11.4955]
})
final_df = district_stats.merge(coords, on='district_id')

fig6 = px.scatter_mapbox(
    final_df, lat='latitude', lon='longitude',
    color='desirability', size='avg_rent_eur',
    hover_name='district_name',
    hover_data={'unemployment_rate': True, 'crime_rate': True, 'avg_rent_eur': True, 'latitude': False, 'longitude': False},
    color_discrete_map=color_map,
    zoom=10, center={'lat': 48.1372, 'lon': 11.5755},
    title='Munich Districts — Desirability Index for New Immigrants 2026',
    height=650
)
fig6.update_layout(mapbox_style='open-street-map')
fig6.show()

## 6. Conclusion

| Category | Districts | Avg Crime | Avg Unemployment | Avg Rent |
|---|---|---|---|---|
| Most Desirable | Sendling-Westpark, Hadern, Untergiesing-Harlaching, Pasing-Obermenzing, Thalkirchen, Allach-Untermenzing | 27.4 | 4.0% | €1,502 |
| Desirable | Bogenhausen, Maxvorstadt, Schwabing-West, Schwabing-Freimann, Neuhausen-Nymphenburg, Au-Haidhausen | 34.9 | 3.6% | €1,864 |
| Semi-Desirable | Altstadt-Lehel | 98.2 | 4.2% | €2,100 |
| Least Desirable | Feldmoching-Hasenbergl, Ramersdorf-Perlach, Milbertshofen-Am Hart + 8 others | 42.8 | 5.7% | €1,395 |

**Key takeaway for new immigrants:** Target the Most Desirable districts — lowest crime, affordable rent, decent employment. Best options: **Sendling-Westpark, Hadern, Pasing-Obermenzing.**

---
**Full article:** [Towards Data Science](#) | [Medium](#)  
**GitHub:** https://github.com/AbiXData/munich-neighbourhood-analysis  
**LinkedIn:** https://www.linkedin.com/in/abitijani/